# Music Annotation Project 59R2
## Component-Invariant REMOVE Challenger

**Runtime:** CPU, High-RAM recommended. Select **Runtime → Run all**.

59R2 is a new development challenger. It does not patch, mutate, rename, or overwrite frozen 59R1. It removes `component_id` completely from the predictor schema, retains component identity only as an audit/join key, adds invariant component geometry and component-level summaries, performs page/source-group-separated out-of-fold development, cross-fits isotonic calibration, freezes a new bundle, compares 59R2 with frozen 59R1 on the complete reviewed cohort, and runs a permanent component-renumbering invariance test.

### Scientific boundary

- `component_id` is prohibited as a predictor.
- `page`, `source_group`, row order, raw coordinates, and component traversal rank are prohibited predictors.
- Clean/truth/outcome fields are labels or evaluation evidence only.
- Existing 59R1 artifacts and 60R2 outputs are read-only comparators.
- The authoritative reviewed mask remains the intervention boundary.
- Promotion authority is **false**. This notebook creates a challenger and evidence package, not a production release.

### 59R2 ordered predictor families

**Existing pixel evidence retained:** observed, normalised and paper values; mask depth; boundary distance; distance to protected evidence; edge indicator; local edge/dark/preserve/remove/ambiguous/horizontal/vertical/compact fractions; calibrated `p_print`.

**Invariant component evidence added:** component area; bounding-box width and height; aspect ratio; fill ratio; page-border contact; component maximum/mean/median depth; component preserve/remove/ambiguous/horizontal/vertical/compact fractions; component mean edge/dark fractions; component mean distance to protected evidence.

`component_id` is written to development and scoring records only as `component_key`, never placed in `X`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy pyarrow scikit-learn joblib opencv-python-headless matplotlib

## 1. Configuration, paths, and fail-closed helpers

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, random, sys, platform, math, zipfile, warnings

import cv2, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')

# Set exact run roots if automatic discovery is inappropriate.
SOURCE_59R1_DEVELOPMENT_RUN_OVERRIDE=None
SOURCE_59R1_FREEZE_RUN_OVERRIDE=None
SOURCE_60R2_RUN_OVERRIDE=None
OBSERVED_PRINT_59P_RUN_OVERRIDE=None

# Candidate roots. The notebook stops rather than guessing if no complete input exists.
DEVELOPMENT_ROOTS=[
    ROOT/'observable_residual_mask_m5_v8_7_development',
    ROOT/'remove_model_development_v8_8',
    ROOT/'remove_classifier_development',
]
FREEZE_59R1_ROOTS=[
    ROOT/'observable_residual_mask_m5_v8_7_deployment_freeze',
    ROOT/'remove_deployment_freeze_v8_8',
    ROOT/'remove_deployment_fit_and_freeze_v8_8',
]
RUN_60R2_ROOTS=[ROOT/'notebook60r2_verified_integrated_runtime',ROOT/'notebook60r2_integrated_runtime']
PRINT_59P_ROOTS=[ROOT/'observed_print_deployment_freeze_v8_2',ROOT/'observed_print_deployment_fit_and_freeze_v8_2']

OUTPUT_ROOT=ROOT/'remove_challenger_59r2_component_invariant'
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_59R2_component_invariant_challenger'
RUN=OUTPUT_ROOT/RUN_ID
INPUTS=RUN/'inputs'; TABLES=RUN/'tables'; MODELS=RUN/'models'; CALIBRATORS=RUN/'calibrators'
BUNDLE=RUN/'bundle'; REPLAY=RUN/'replay'; AUDIT=RUN/'audit'; COMPARISON=RUN/'comparison_59R1'; UPLOAD=RUN/'upload'; CHARTS=RUN/'charts'
for p in [RUN,INPUTS,TABLES,MODELS,CALIBRATORS,BUNDLE,REPLAY,AUDIT,COMPARISON,UPLOAD,CHARTS]: p.mkdir(parents=True,exist_ok=False)

SEED=42
N_SPLITS=10
REMOVE_THRESHOLD=0.995
REMOVE_CONNECTIVITY=8
REMOVE_MINIMUM_RUN=2
REMOVE_REGION_MEAN_THRESHOLD=0.995
RANDOM_RENUMBERING_SEEDS=[11,23,37,53,71,89,107,131,151,173]
EXPECTED_REVIEWED_PAGES=151
REQUIRE_EXACT_REVIEWED_COHORT=True

random.seed(SEED); np.random.seed(SEED)


def require(x,message):
    if not bool(x): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def atomic_json(path,value): Path(path).write_text(json.dumps(value,indent=2,sort_keys=True,default=str))

def latest_complete(roots,override,required):
    if override is not None:
        p=Path(override); p=p.parent if p.name in {'bundle','artefacts'} else p
        require(all((p/r).is_file() for r in required),f'Incomplete override: {p}')
        return p
    hits=[]
    for root in roots:
        if root.exists():
            for p in root.rglob('*'):
                if p.is_dir() and all((p/r).is_file() for r in required): hits.append(p)
    require(hits,f'No complete run found below {[str(x) for x in roots]}')
    return max(hits,key=lambda p:p.stat().st_mtime)

print('59R2 output:',RUN)

## 2. Predictor firewall and frozen schema

In [ ]:
IDENTIFIERS=['source_group','page','component_key','component_id','y','x','row_id']
PROHIBITED_EXACT=set(IDENTIFIERS+['component','component_rank','component_order','label_id'])
PROHIBITED_PATTERNS=['clean','truth','target','outcome','failure','precision','recall','mae','safe_remove','p_remove','raw_remove_probability','structural_safe']

PIXEL_PREDICTORS=[
 'observed_value','normalised_value','paper_value','mask_depth','boundary_distance',
 'distance_to_protected','edge','local_edge_fraction','local_dark_fraction',
 'local_preserve_fraction','local_remove_fraction','local_ambiguous_fraction',
 'local_horizontal_fraction','local_vertical_fraction','local_compact_fraction','p_print'
]
COMPONENT_PREDICTORS=[
 'component_area','component_width','component_height','component_aspect_ratio','component_fill_ratio',
 'component_touches_page_border','component_maximum_mask_depth','component_mean_mask_depth','component_median_mask_depth',
 'component_preserve_fraction','component_remove_fraction','component_ambiguous_fraction',
 'component_horizontal_fraction','component_vertical_fraction','component_compact_fraction',
 'component_mean_edge_fraction','component_mean_dark_fraction','component_mean_distance_to_protected'
]
PREDICTORS=PIXEL_PREDICTORS+COMPONENT_PREDICTORS
require('component_id' not in PREDICTORS and 'component_key' not in PREDICTORS,'Identifier leakage in 59R2 schema.')
require(len(PREDICTORS)==len(set(PREDICTORS)),'Duplicate 59R2 predictors.')
for c in PREDICTORS:
    require(c not in PROHIBITED_EXACT and not any(p in c.lower() for p in PROHIBITED_PATTERNS),f'Forbidden predictor: {c}')

schema={
 'schema_version':'59R2-component-invariant-v1',
 'ordered_predictors':PREDICTORS,
 'identifier_columns':['source_group','page','component_key','y','x'],
 'prohibited_predictors':sorted(PROHIBITED_EXACT),
 'component_identity_role':'audit_join_key_only',
 'component_id_is_predictor':False,
 'predictor_count':len(PREDICTORS),
}
atomic_json(INPUTS/'predictor_schema.json',schema)
print('59R2 predictors:',len(PREDICTORS))

## 3. Resolve development rows, frozen 59R1, and 60R2 cohort

In [ ]:
# Development run must contain a row-level labelled table. Accepted filenames are resolved explicitly.
def resolve_development_run():
    if SOURCE_59R1_DEVELOPMENT_RUN_OVERRIDE is not None: return Path(SOURCE_59R1_DEVELOPMENT_RUN_OVERRIDE)
    candidates=[]
    names=['tables/remove_development_pixels.parquet','tables/pixel_training_dataset.parquet','tables/remove_training_dataset.parquet','remove_development_pixels.parquet']
    for root in DEVELOPMENT_ROOTS:
        if root.exists():
            for p in root.rglob('*'):
                if p.is_dir() and any((p/n).is_file() for n in names): candidates.append(p)
    require(candidates,'No 59R1 development run with a recognised labelled pixel table was found. Set SOURCE_59R1_DEVELOPMENT_RUN_OVERRIDE.')
    return max(candidates,key=lambda p:p.stat().st_mtime)

def resolve_first(run,names):
    hits=[run/n for n in names if (run/n).is_file()]
    require(len(hits)==1,f'Expected exactly one recognised file in {run}; found {hits}')
    return hits[0]

DEV_RUN=resolve_development_run()
DEV_PATH=resolve_first(DEV_RUN,['tables/remove_development_pixels.parquet','tables/pixel_training_dataset.parquet','tables/remove_training_dataset.parquet','remove_development_pixels.parquet'])

R59=latest_complete(FREEZE_59R1_ROOTS,SOURCE_59R1_FREEZE_RUN_OVERRIDE,[
 'bundle/remove_model.joblib','bundle/remove_calibrator.joblib','bundle/predictor_schema.json','bundle/policy.json','replay/replay_test.json'])
R60=latest_complete(RUN_60R2_ROOTS,SOURCE_60R2_RUN_OVERRIDE,[
 'verdict.json','tables/pixel_predictions.parquet','tables/page_summary.csv'])

source_files={'development_table':DEV_PATH,'59r1_model':R59/'bundle'/'remove_model.joblib','59r1_calibrator':R59/'bundle'/'remove_calibrator.joblib','59r1_schema':R59/'bundle'/'predictor_schema.json','59r1_policy':R59/'bundle'/'policy.json','59r1_replay':R59/'replay'/'replay_test.json','60r2_pixels':R60/'tables'/'pixel_predictions.parquet','60r2_page_summary':R60/'tables'/'page_summary.csv','60r2_verdict':R60/'verdict.json'}
atomic_json(AUDIT/'source_hashes_before.json',{k:{'path':str(v),'sha256':sha256_file(v)} for k,v in source_files.items()})
print('Development:',DEV_RUN) ; print('59R1:',R59); print('60R2:',R60)

## 4. Load and normalise the labelled development dataset

In [ ]:
dev=pd.read_parquet(DEV_PATH)
require(len(dev)>0,'Development dataset is empty.')

# Fail-closed aliases. No inferred label semantics beyond these explicit names.
label_candidates=['remove_target','target_remove','is_remove','remove_label','label']
label_hits=[c for c in label_candidates if c in dev.columns]
require(len(label_hits)==1,f'Expected one explicit REMOVE label column, found {label_hits}')
LABEL=label_hits[0]

group_candidates=['source_group','score_source_group','group']
group_hits=[c for c in group_candidates if c in dev.columns]
require(len(group_hits)>=1,'Development data must contain source_group or equivalent.')
SOURCE_GROUP=group_hits[0]
require('page' in dev.columns,'Development data must contain page.')

# Component identity is retained only for aggregation/joining.
if 'component_key' not in dev.columns:
    if 'component_id' in dev.columns: dev['component_key']=dev['component_id'].astype(str)
    elif 'component' in dev.columns: dev['component_key']=dev['component'].astype(str)
    else: raise RuntimeError('Development data requires a component identity column for audit/join and invariant feature aggregation.')

dev['page']=dev.page.astype(str); dev['source_group']=dev[SOURCE_GROUP].astype(str); dev['component_key']=dev.component_key.astype(str)
y=pd.to_numeric(dev[LABEL],errors='raise').astype(int)
require(set(y.unique()).issubset({0,1}) and y.nunique()==2,f'Invalid binary REMOVE label values: {sorted(y.unique())}')

# Existing pixel predictors must be present. Invariant component fields may be rebuilt below.
missing_pixel=[c for c in PIXEL_PREDICTORS if c not in dev.columns]
require(not missing_pixel,f'Development table lacks required pixel predictors: {missing_pixel}')
require({'y','x'}.issubset(dev.columns),'Development table requires y/x for immutable component summaries.')
print('Development rows:',len(dev),'pages:',dev.page.nunique(),'source groups:',dev.source_group.nunique(),'positive prevalence:',float(y.mean()))

## 5. Rebuild legitimate invariant component geometry and summaries

In [ ]:
def add_component_features(frame):
    f=frame.copy()
    rows=[]
    H_col=next((c for c in ['page_height','height','image_height'] if c in f.columns),None)
    W_col=next((c for c in ['page_width','width','image_width'] if c in f.columns),None)
    for (page,ck),g in f.groupby(['page','component_key'],sort=False):
        ys=pd.to_numeric(g.y,errors='raise').to_numpy(int); xs=pd.to_numeric(g.x,errors='raise').to_numpy(int)
        area=len(g); x0,x1=int(xs.min()),int(xs.max())+1; y0,y1=int(ys.min()),int(ys.max())+1
        w=x1-x0; h=y1-y0
        page_h=int(g[H_col].iloc[0]) if H_col else int(f.loc[f.page==page,'y'].max())+1
        page_w=int(g[W_col].iloc[0]) if W_col else int(f.loc[f.page==page,'x'].max())+1
        def mean(c): return float(pd.to_numeric(g[c],errors='coerce').mean())
        rows.append({
          'page':page,'component_key':ck,
          'component_area':int(area),'component_width':int(w),'component_height':int(h),
          'component_aspect_ratio':float(max(w,h)/max(min(w,h),1)),
          'component_fill_ratio':float(area/max(w*h,1)),
          'component_touches_page_border':int(x0==0 or y0==0 or x1>=page_w or y1>=page_h),
          'component_maximum_mask_depth':float(pd.to_numeric(g.mask_depth,errors='coerce').max()),
          'component_mean_mask_depth':mean('mask_depth'),'component_median_mask_depth':float(pd.to_numeric(g.mask_depth,errors='coerce').median()),
          'component_preserve_fraction':mean('local_preserve_fraction'),'component_remove_fraction':mean('local_remove_fraction'),
          'component_ambiguous_fraction':mean('local_ambiguous_fraction'),'component_horizontal_fraction':mean('local_horizontal_fraction'),
          'component_vertical_fraction':mean('local_vertical_fraction'),'component_compact_fraction':mean('local_compact_fraction'),
          'component_mean_edge_fraction':mean('local_edge_fraction'),'component_mean_dark_fraction':mean('local_dark_fraction'),
          'component_mean_distance_to_protected':mean('distance_to_protected'),
        })
    comp=pd.DataFrame(rows)
    require(not comp.duplicated(['page','component_key']).any(),'Duplicate component feature records.')
    return f.drop(columns=[c for c in COMPONENT_PREDICTORS if c in f.columns],errors='ignore').merge(comp,on=['page','component_key'],validate='many_to_one'),comp

dev,development_components=add_component_features(dev)
development_components.to_parquet(TABLES/'development_component_features.parquet',index=False)
missing=[c for c in PREDICTORS if c not in dev.columns]; require(not missing,f'59R2 predictors missing after rebuild: {missing}')
X=dev[PREDICTORS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
require(not np.isinf(X.to_numpy(float)).any(),'Infinite development predictor value.')

# Predictor firewall: exact schema and no hidden identifiers.
require(list(X.columns)==PREDICTORS,'Ordered schema mismatch.')
for forbidden in PROHIBITED_EXACT:
    require(forbidden not in X.columns,f'Forbidden predictor entered X: {forbidden}')
shutil.copy2(DEV_PATH,INPUTS/DEV_PATH.name)
dev[['source_group','page','component_key','y','x',LABEL]+PREDICTORS].to_parquet(TABLES/'59r2_development_dataset.parquet',index=False)
print('Built invariant component predictors:',len(development_components))

## 6. Page/source-group-separated fold map

In [ ]:
# Prefer source-group separation. Page identity is additionally asserted disjoint.
groups=dev.source_group.astype(str)
unique_groups=groups.nunique()
require(unique_groups>=3,'At least three source groups are required.')
splitter=LeaveOneGroupOut() if unique_groups<=10 else GroupKFold(n_splits=min(N_SPLITS,unique_groups))
fold=np.full(len(dev),-1,int); split_rows=[]
for f,(tr,te) in enumerate(splitter.split(X,y,groups),1):
    train_groups=set(groups.iloc[tr]); test_groups=set(groups.iloc[te]); train_pages=set(dev.page.iloc[tr]); test_pages=set(dev.page.iloc[te])
    require(not train_groups&test_groups,f'Fold {f}: source-group leakage.')
    require(not train_pages&test_pages,f'Fold {f}: page leakage.')
    fold[te]=f
    split_rows.append({'fold':f,'train_rows':len(tr),'test_rows':len(te),'train_pages':len(train_pages),'test_pages':len(test_pages),'train_groups':len(train_groups),'test_groups':len(test_groups),'test_group_names':'|'.join(sorted(test_groups))})
require((fold>0).all(),'Incomplete fold assignment.')
dev['fold']=fold
pd.DataFrame(split_rows).to_csv(INPUTS/'fold_map.csv',index=False)
print(pd.DataFrame(split_rows).to_string(index=False))

## 7. Candidate models and source-group OOF development

In [ ]:
MODELS={
 'logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]),
 'forest':Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=600,min_samples_leaf=10,max_features='sqrt',class_weight='balanced_subsample',random_state=SEED,n_jobs=-1))]),
 'hist_gradient':Pipeline([('impute',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=2.0,random_state=SEED))]),
}

summaries=[]; fold_results=[]; raw_by_model={}
for name,template in MODELS.items():
    raw=np.full(len(dev),np.nan)
    for f in sorted(np.unique(fold)):
        tr=np.where(fold!=f)[0]; te=np.where(fold==f)[0]
        require(y.iloc[tr].nunique()==2,f'{name} fold {f}: one-class training set.')
        m=clone(template).fit(X.iloc[tr],y.iloc[tr]); raw[te]=m.predict_proba(X.iloc[te])[:,1]
        yt=y.iloc[te].to_numpy(); pt=raw[te]
        fold_results.append({'model':name,'fold':int(f),'test_rows':len(te),'test_pages':dev.page.iloc[te].nunique(),'test_groups':dev.source_group.iloc[te].nunique(),'roc_auc':roc_auc_score(yt,pt) if len(np.unique(yt))==2 else np.nan,'average_precision':average_precision_score(yt,pt),'brier':brier_score_loss(yt,pt)})
    require(np.isfinite(raw).all(),f'{name}: incomplete OOF predictions.')
    raw_by_model[name]=raw
    summaries.append({'model':name,'roc_auc':roc_auc_score(y,raw),'average_precision':average_precision_score(y,raw),'brier':brier_score_loss(y,raw)})
summary_df=pd.DataFrame(summaries).sort_values(['average_precision','roc_auc'],ascending=False)
summary_df.to_csv(TABLES/'model_comparison.csv',index=False); pd.DataFrame(fold_results).to_csv(TABLES/'fold_results.csv',index=False)
BEST=str(summary_df.iloc[0].model)
print('Selected model family:',BEST); display(summary_df)

## 8. Cross-fitted isotonic calibration and frozen decision policy

In [ ]:
raw=raw_by_model[BEST]
calibrated=np.full(len(dev),np.nan)
for f in sorted(np.unique(fold)):
    tr=np.where(fold!=f)[0]; te=np.where(fold==f)[0]
    iso=IsotonicRegression(out_of_bounds='clip').fit(raw[tr],y.iloc[tr].to_numpy())
    calibrated[te]=iso.predict(raw[te])
require(np.isfinite(calibrated).all(),'Incomplete cross-fitted calibration.')

dev['raw_59r2_oof']=raw; dev['p_remove_59r2_oof']=calibrated; dev['provisional_remove_59r2']=calibrated>=REMOVE_THRESHOLD
dev[['source_group','page','component_key','y','x','fold',LABEL,'raw_59r2_oof','p_remove_59r2_oof','provisional_remove_59r2']].to_parquet(TABLES/'59r2_oof_predictions.parquet',index=False)

metrics={'model':BEST,'roc_auc_raw':float(roc_auc_score(y,raw)),'average_precision_raw':float(average_precision_score(y,raw)),'brier_raw':float(brier_score_loss(y,raw)),'brier_calibrated':float(brier_score_loss(y,calibrated)),'threshold':REMOVE_THRESHOLD,'selected_pixels':int((calibrated>=REMOVE_THRESHOLD).sum()),'selected_positive':int(((calibrated>=REMOVE_THRESHOLD)&(y.to_numpy()==1)).sum()),'selected_negative':int(((calibrated>=REMOVE_THRESHOLD)&(y.to_numpy()==0)).sum())}
atomic_json(RUN/'development_metrics.json',metrics)
print(json.dumps(metrics,indent=2))

## 9. Refit chosen 59R2 estimator and calibrator on permitted development evidence

In [ ]:
final_model=clone(MODELS[BEST]).fit(X,y)
# Calibration is fitted to OOF raw scores, never in-sample fitted scores.
final_calibrator=IsotonicRegression(out_of_bounds='clip').fit(raw,y.to_numpy())
joblib.dump(final_model,BUNDLE/'remove_model.joblib')
joblib.dump(final_calibrator,BUNDLE/'remove_calibrator.joblib')
shutil.copy2(INPUTS/'predictor_schema.json',BUNDLE/'predictor_schema.json')

policy_payload={
 'policy_version':'59R2-component-invariant-v1','policy_id':None,'development_only':True,'promotion_authority':False,
 'threshold':REMOVE_THRESHOLD,'remove_threshold':REMOVE_THRESHOLD,'connectivity':REMOVE_CONNECTIVITY,
 'minimum_run':REMOVE_MINIMUM_RUN,'region_mean_threshold':REMOVE_REGION_MEAN_THRESHOLD,
 'safe_protect_enabled':False,'component_id_is_predictor':False,'component_identity_role':'audit_join_key_only',
 'model_family':BEST,'calibration':'isotonic fitted to source-group OOF raw scores','ordered_predictors':PREDICTORS,
}
policy_payload['policy_id']='59R2-'+object_hash(policy_payload)[:16]
atomic_json(BUNDLE/'policy.json',policy_payload)
print('Policy ID:',policy_payload['policy_id'])

## 10. Exact replay pack

In [ ]:
# Deterministic multi-page replay sample.
rng=np.random.default_rng(SEED)
replay_idx=[]
for page,g in dev.groupby('page'):
    idx=g.index.to_numpy(); replay_idx.extend(rng.choice(idx,min(25,len(idx)),replace=False).tolist())
replay_idx=np.array(sorted(replay_idx),int)
replay_X=X.loc[replay_idx].reset_index(drop=True)
replay_meta=dev.loc[replay_idx,['source_group','page','component_key','y','x']].reset_index(drop=True)
replay_raw=np.asarray(final_model.predict_proba(replay_X)[:,1],np.float64)
replay_prob=np.asarray(final_calibrator.predict(replay_raw),np.float64)
replay_decision=replay_prob>=REMOVE_THRESHOLD
replay_meta.to_csv(REPLAY/'replay_rows.csv',index=False); replay_X.to_parquet(REPLAY/'replay_predictors.parquet',index=False)
np.savez_compressed(REPLAY/'replay_expected.npz',raw=replay_raw,probability=replay_prob,decision=replay_decision.astype(np.uint8))

m2=joblib.load(BUNDLE/'remove_model.joblib'); c2=joblib.load(BUNDLE/'remove_calibrator.joblib')
r2=np.asarray(m2.predict_proba(replay_X)[:,1],np.float64); p2=np.asarray(c2.predict(r2),np.float64); d2=p2>=REMOVE_THRESHOLD
replay_result={'passed':bool(np.array_equal(r2,replay_raw) and np.array_equal(p2,replay_prob) and np.array_equal(d2,replay_decision)),'model_reload_exact':bool(np.array_equal(r2,replay_raw)),'calibrator_reload_exact':bool(np.array_equal(p2,replay_prob)),'decision_reload_exact':bool(np.array_equal(d2,replay_decision)),'rows':len(replay_X),'policy_id':policy_payload['policy_id']}
atomic_json(REPLAY/'replay_test.json',replay_result); require(replay_result['passed'],'59R2 replay failed.')
print(json.dumps(replay_result,indent=2))

## 11. Score complete 60R2 reviewed cohort with 59R2 and compare frozen 59R1

In [ ]:
cohort=pd.read_parquet(source_files['60r2_pixels']).copy()
require(len(cohort)>0 and {'page','y','x','component_id','raw_remove_probability','p_remove','safe_remove'}.issubset(cohort.columns),'60R2 cohort lacks required fields.')
cohort['page']=cohort.page.astype(str); cohort['component_key']=cohort.component_id.astype(str)
if REQUIRE_EXACT_REVIEWED_COHORT: require(cohort.page.nunique()==EXPECTED_REVIEWED_PAGES,f'Expected {EXPECTED_REVIEWED_PAGES} reviewed pages; found {cohort.page.nunique()}.')
missing=[c for c in PIXEL_PREDICTORS if c not in cohort.columns]; require(not missing,f'60R2 cohort lacks pixel predictors: {missing}')
cohort,cohort_components=add_component_features(cohort)
Xc=cohort[PREDICTORS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
raw2=np.asarray(final_model.predict_proba(Xc)[:,1],np.float64); prob2=np.asarray(final_calibrator.predict(raw2),np.float64)
cohort['raw_remove_59r2']=raw2; cohort['p_remove_59r2']=prob2

# Frozen page-global coherence, using 60R2 page extents from coordinates.
def coherent(frame,prob):
    h=int(frame.y.max())+1; w=int(frame.x.max())+1
    candidate=np.zeros((h,w),np.uint8); hit=np.asarray(prob)>=REMOVE_THRESHOLD
    candidate[frame.loc[hit,'y'].astype(int),frame.loc[hit,'x'].astype(int)]=1
    n,labels,stats,_=cv2.connectedComponentsWithStats(candidate,REMOVE_CONNECTIVITY)
    pmap=np.full((h,w),np.nan,np.float32); pmap[frame.y.astype(int),frame.x.astype(int)]=np.asarray(prob,np.float32)
    safe=np.zeros((h,w),bool)
    for cid in range(1,n):
        region=labels==cid; vals=pmap[region]; vals=vals[np.isfinite(vals)]
        if len(vals)==int(region.sum()) and int(region.sum())>=REMOVE_MINIMUM_RUN and float(vals.mean())>=REMOVE_REGION_MEAN_THRESHOLD and float(vals.min())>=REMOVE_THRESHOLD: safe[region]=True
    return safe

safe59r2=np.zeros(len(cohort),bool); page_rows=[]
for page,g in cohort.groupby('page',sort=False):
    safe=coherent(g,g.p_remove_59r2.to_numpy())
    vals=safe[g.y.astype(int),g.x.astype(int)]
    safe59r2[g.index.to_numpy()]=vals
    old=g.safe_remove.astype(bool).to_numpy()
    page_rows.append({'page':page,'pixels':len(g),'safe_remove_59r1':int(old.sum()),'safe_remove_59r2':int(vals.sum()),'delta':int(vals.sum()-old.sum()),'changed_decisions':int(np.sum(vals!=old))})
cohort['safe_remove_59r2']=safe59r2
cohort[['page','component_key','y','x','raw_remove_probability','p_remove','safe_remove','raw_remove_59r2','p_remove_59r2','safe_remove_59r2']].to_parquet(COMPARISON/'59r1_vs_59r2_pixel_decisions.parquet',index=False)
page_compare=pd.DataFrame(page_rows); page_compare.to_csv(COMPARISON/'59r1_vs_59r2_by_page.csv',index=False)
comparison_summary={'pages':int(cohort.page.nunique()),'59r1_safe_remove_pixels':int(cohort.safe_remove.astype(bool).sum()),'59r2_safe_remove_pixels':int(cohort.safe_remove_59r2.sum()),'safe_remove_delta':int(cohort.safe_remove_59r2.sum()-cohort.safe_remove.astype(bool).sum()),'changed_pixel_decisions':int(np.sum(cohort.safe_remove_59r2!=cohort.safe_remove.astype(bool).to_numpy()))}
atomic_json(COMPARISON/'comparison_summary.json',comparison_summary)
print(json.dumps(comparison_summary,indent=2))

## 12. Permanent component-renumbering invariance test

In [ ]:
# Since component identity is excluded from X, equivalent renumberings must produce exact equality.
baseline_raw=raw2.copy(); baseline_prob=prob2.copy(); baseline_decision=cohort.safe_remove_59r2.to_numpy().copy()
invariance_rows=[]

for seed in RANDOM_RENUMBERING_SEEDS:
    rng=np.random.default_rng(seed)
    altered=cohort[['page','component_key']].copy()
    new_keys=[]
    for page,g in cohort.groupby('page',sort=False):
        keys=sorted(g.component_key.unique()); perm=keys.copy(); rng.shuffle(perm); mapping=dict(zip(keys,perm))
        new_keys.extend(g.component_key.map(mapping).tolist())
    # Reattribute keys in original row order safely.
    renamed=cohort.copy(); renamed['component_key']=pd.Series(new_keys,index=cohort.groupby('page',sort=False).apply(lambda z:z.index.tolist(),include_groups=False).explode().astype(int)).sort_index().to_numpy()
    # No rebuilding of geometry is needed: semantic component membership and invariant features are fixed.
    Xr=renamed[PREDICTORS].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
    rr=np.asarray(final_model.predict_proba(Xr)[:,1],np.float64); pp=np.asarray(final_calibrator.predict(rr),np.float64)
    require(np.array_equal(Xr.to_numpy(),Xc.to_numpy()),f'Seed {seed}: predictor matrix changed under identifier renumbering.')
    require(np.array_equal(rr,baseline_raw),f'Seed {seed}: raw scores changed.')
    require(np.array_equal(pp,baseline_prob),f'Seed {seed}: probabilities changed.')
    # Coherent output replay remains exact because coordinates and probabilities are unchanged.
    invariance_rows.append({'seed':seed,'predictor_matrix_exact':True,'raw_score_exact':True,'probability_exact':True,'threshold_decision_exact':True,'coherent_output_exact':True,'max_abs_raw_difference':0.0,'max_abs_probability_difference':0.0,'changed_decisions':0})

invariance=pd.DataFrame(invariance_rows); invariance.to_csv(AUDIT/'component_renumbering_invariance.csv',index=False)
invariance_verdict={'passed':bool(invariance[['predictor_matrix_exact','raw_score_exact','probability_exact','threshold_decision_exact','coherent_output_exact']].all().all()),'requirement':'exactly zero score, probability, decision and output change','tested_seeds':RANDOM_RENUMBERING_SEEDS,'component_id_predictor_present':False,'changed_decisions_total':int(invariance.changed_decisions.sum())}
atomic_json(AUDIT/'component_renumbering_invariance_verdict.json',invariance_verdict); require(invariance_verdict['passed'],'Permanent renumbering invariance test failed.')
print(json.dumps(invariance_verdict,indent=2))

## 13. Freeze manifest, compatibility identity, and source immutability

In [ ]:
# Add hashes after all primary objects exist.
policy=read_policy=json.loads((BUNDLE/'policy.json').read_text())
policy.update({'model_sha256':sha256_file(BUNDLE/'remove_model.joblib'),'calibrator_sha256':sha256_file(BUNDLE/'remove_calibrator.joblib'),'schema_sha256':sha256_file(BUNDLE/'predictor_schema.json'),'replay_sha256':sha256_file(REPLAY/'replay_test.json')})
compatibility_payload={'schema_sha256':policy['schema_sha256'],'threshold':REMOVE_THRESHOLD,'connectivity':REMOVE_CONNECTIVITY,'minimum_run':REMOVE_MINIMUM_RUN,'region_mean_threshold':REMOVE_REGION_MEAN_THRESHOLD,'component_id_is_predictor':False}
policy['compatibility_id']='59R2-COMPAT-'+object_hash(compatibility_payload)[:16]
atomic_json(BUNDLE/'policy.json',policy)

runtime_contract={'policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],'ordered_predictors':PREDICTORS,'identifier_columns':['source_group','page','component_key','y','x'],'component_identity_role':'audit_join_key_only','component_renumbering_invariance_required':True,'threshold':REMOVE_THRESHOLD,'connectivity':REMOVE_CONNECTIVITY,'minimum_run':REMOVE_MINIMUM_RUN,'region_mean_threshold':REMOVE_REGION_MEAN_THRESHOLD,'development_grouping':'source_group with page-disjoint assertion','promotion_authority':False}
atomic_json(BUNDLE/'runtime_contract.json',runtime_contract)

freeze_files=[BUNDLE/'remove_model.joblib',BUNDLE/'remove_calibrator.joblib',BUNDLE/'predictor_schema.json',BUNDLE/'policy.json',BUNDLE/'runtime_contract.json',REPLAY/'replay_test.json',REPLAY/'replay_rows.csv',REPLAY/'replay_predictors.parquet',REPLAY/'replay_expected.npz']
freeze={'created_utc':datetime.now(timezone.utc).isoformat(),'bundle_version':'59R2-component-invariant-v1','policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],'development_bundle_complete':True,'promotion_authority':False,'objects':[{'relative_path':str(p.relative_to(RUN)),'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in freeze_files]}
atomic_json(BUNDLE/'freeze_manifest.json',freeze)

source_hashes_after={k:sha256_file(v) for k,v in source_files.items()}
source_hashes_before={k:v['sha256'] for k,v in json.loads((AUDIT/'source_hashes_before.json').read_text()).items()}
require(source_hashes_after==source_hashes_before,'A source artifact changed during 59R2.')
atomic_json(AUDIT/'source_hashes_after.json',{k:{'path':str(source_files[k]),'sha256':v} for k,v in source_hashes_after.items()})

## 14. Challenger verdict and concise comparison charts

In [ ]:
verdict={
 'verdict':'CHALLENGER_BUILT',
 'promotion_authority':False,
 'policy_id':policy['policy_id'],'compatibility_id':policy['compatibility_id'],
 'component_id_removed_from_predictors':True,
 'component_identity_audit_key_only':True,
 'page_source_group_separated_development':True,
 'model_refit':True,'isotonic_calibrator_refit_from_oof_scores':True,
 'exact_replay_passed':replay_result['passed'],
 'renumbering_invariance_passed':invariance_verdict['passed'],
 'complete_reviewed_cohort_pages':int(cohort.page.nunique()),
 '59r1_safe_remove_pixels':comparison_summary['59r1_safe_remove_pixels'],
 '59r2_safe_remove_pixels':comparison_summary['59r2_safe_remove_pixels'],
 'changed_pixel_decisions':comparison_summary['changed_pixel_decisions'],
 'source_artifacts_unchanged':True,
 'next_gate':'independent safety/benefit review and documented promotion decision; no automatic promotion',
}
atomic_json(RUN/'59r2_verdict.json',verdict)

q=page_compare.sort_values('delta')
plt.figure(figsize=(11,max(6,.055*len(q))))
plt.barh(q.page,q.delta); plt.axvline(0,color='black',linewidth=1); plt.xlabel('59R2 minus 59R1 coherent SAFE_REMOVE pixels'); plt.title('59R2 versus frozen 59R1 by reviewed page'); plt.tight_layout(); plt.savefig(CHARTS/'59r2_vs_59r1_safe_remove_delta_by_page.png',dpi=160); plt.close()

lines=['59R2 COMPONENT-INVARIANT REMOVE CHALLENGER','',f'Policy ID: {policy["policy_id"]}',f'Compatibility ID: {policy["compatibility_id"]}',f'Model family: {BEST}',f'Development rows: {len(dev)}',f'Development pages: {dev.page.nunique()}',f'Development source groups: {dev.source_group.nunique()}',f'Ordered predictors: {len(PREDICTORS)}',f'component_id predictor: False',f'Exact replay: {replay_result["passed"]}',f'Renumbering invariance: {invariance_verdict["passed"]}',f'Reviewed cohort pages: {cohort.page.nunique()}',f'59R1 SAFE_REMOVE pixels: {comparison_summary["59r1_safe_remove_pixels"]}',f'59R2 SAFE_REMOVE pixels: {comparison_summary["59r2_safe_remove_pixels"]}',f'Changed decisions: {comparison_summary["changed_pixel_decisions"]}','','59R2 is a new challenger, not a patch to 59R1 and not production-authorised.']
(RUN/'executive_summary.txt').write_text(chr(10).join(lines)+chr(10)); print(chr(10).join(lines))

## 15. Standard Final Upload Package

In [ ]:
curated=[
 RUN/'executive_summary.txt',RUN/'59r2_verdict.json',RUN/'development_metrics.json',
 BUNDLE/'policy.json',BUNDLE/'predictor_schema.json',BUNDLE/'runtime_contract.json',BUNDLE/'freeze_manifest.json',
 REPLAY/'replay_test.json',AUDIT/'component_renumbering_invariance.csv',AUDIT/'component_renumbering_invariance_verdict.json',
 AUDIT/'source_hashes_before.json',AUDIT/'source_hashes_after.json',TABLES/'model_comparison.csv',TABLES/'fold_results.csv',
 COMPARISON/'comparison_summary.json',COMPARISON/'59r1_vs_59r2_by_page.csv',CHARTS/'59r2_vs_59r1_safe_remove_delta_by_page.png',
]
manifest=[]
for src in curated:
    require(src.is_file(),f'Missing upload artifact: {src}')
    rel=src.relative_to(RUN); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
    manifest.append({'relative_path':str(rel),'bytes':src.stat().st_size,'sha256':sha256_file(src)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('59R2 component-invariant REMOVE challenger review package.'+chr(10)+'The complete model and calibrator remain in the run bundle and are hash-addressed by freeze_manifest.json.'+chr(10)+'This package has no production-promotion authority.'+chr(10))
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+chr(10))
zip_path=RUN/'59R2_Component_Invariant_REMOVE_Challenger_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): zf.write(p,p.relative_to(UPLOAD))
require(zip_path.is_file() and zip_path.stat().st_size>0,'Upload ZIP failed.')
print('Upload ZIP:',zip_path)